In [124]:
import pandas as pd
import unicodedata

results = pd.read_csv("../datasets/results.csv")
drivers = pd.read_csv("../datasets/drivers.csv")
constructors = pd.read_csv("../datasets/constructors.csv")
races = pd.read_csv("../datasets/races.csv")

In [125]:
df = results.merge(drivers, on="driverId")
df = df.merge(constructors, on="constructorId")
df = df.merge(races, on="raceId")

In [126]:
df = df.rename(columns={
    "surname": "piloto",
    "name_x": "equipe",
    "name_y": "corrida",
    "grid": "largada",
    "position": "posicao",
    "points": "pontos",
    "year": "ano",
    "nationality_x": "nacionalidade"
})

In [127]:
df_final = df[[
    "piloto",
    "equipe",
    "corrida",
    "largada",   
    "posicao",
    "pontos",
    "ano",
    "nacionalidade"  
]]

df_final = df_final[df_final["ano"] >= 2018]
df_final["posicao"] = df_final["posicao"].replace("\\N", 0)

In [128]:
def clean(text):
    text = str(text).lower().replace(" ", "_")
    
    text = unicodedata.normalize('NFKD', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    if text.endswith("_"):
        text = text[:-1]
    
    return text

for col in ["piloto", "equipe", "corrida", "nacionalidade"]:
    df_final[col] = df_final[col].apply(clean)
    
df_final["corrida"] = df_final["corrida"].apply(
    lambda x: "gp_" + x if x[0].isdigit() else x
)

df_final.head()

,piloto,equipe,corrida,largada,posicao,pontos,ano,nacionalidade
23777,vettel,ferrari,australian_grand_prix,3,1,25.0,2018,german
23778,hamilton,mercedes,australian_grand_prix,1,2,18.0,2018,british
23779,raikkonen,ferrari,australian_grand_prix,2,3,15.0,2018,finnish
23780,ricciardo,red_bull,australian_grand_prix,8,4,12.0,2018,australian
23781,alonso,mclaren,australian_grand_prix,10,5,10.0,2018,spanish


In [129]:
with open("../prolog/base.pl", "w") as f:
    for _, row in df_final.iterrows():
        linha = f"corrida({row['piloto']}, {row['equipe']}, {row['corrida']}, {row['largada']}, {row['posicao']}, {row['pontos']}, {row['ano']}, {row['nacionalidade']}).\n"
        f.write(linha)